In [0]:
# ============================================================
# CONFIGURACIÓN GLOBAL — Vermont EWS
# Esta celda hace el notebook autocontenido para Workflows
# ============================================================

import re, os
from datetime import datetime

# Rutas principales
BRONZE  = "/Volumes/workspace/vermont/bronze"
TRUSTED = "/Volumes/workspace/vermont/trusted"
SILVER  = "/Volumes/workspace/vermont/silver"
PRIVADO = "/Volumes/workspace/vermont/privado"

# Materias
GROUPS = [
    'Science', 'I_and_S', 'Mathematics', 'English',
    'Lengua_Castellana', 'Mandarin', 'Financial_Maths',
    'ICT_STEM', 'Physical_Education', 'Research_Methodology'
]

# Features del modelo
FEATURES_CLASIF = (
    [f'{g}_T1' for g in GROUPS] +
    [f'{g}_T2' for g in GROUPS] +
    [f'{g}_delta' for g in GROUPS] +
    ['total_absences', 'absence_class', 'late', 'early_leave',
     'n_f1', 'n_f2', 'n_fault_types',
     'tendencia_general', 'avg_T1', 'avg_T2',
     'n_bajo_T1', 'n_bajo_T2', 'delta_materias_bajo',
     'dispersion_T2', 'ratio_ausencia_clase',
     'indice_disciplinario', 'min_nota_T2', 'min_nota_T1']
)

# Umbral de producción del clasificador
UMBRAL_PRODUCCION = 0.30

# Año académico actual
ANO_ACTUAL   = '25_26'
ANO_ANTERIOR = '24_25'

# ── Fecha de corte ──
def detectar_fecha_corte():
    RAW_25 = f"{BRONZE}/raw/25_26"
    fechas = []
    for f in os.listdir(RAW_25):
        m = re.search(r'_(\d{2})_(\d{2})_(\d{2})\.xls$', f)
        if m:
            d, mo, y = m.groups()
            fechas.append((datetime(int(f"20{y}"), int(mo), int(d)), f"{d}_{mo}_{y}"))
    return max(fechas, key=lambda x: x[0])[1] if fechas else datetime.now().strftime("%d_%m_%y")

FECHA_TAG = detectar_fecha_corte()

# ── Función de archivado histórico ──
def archivar_antes_de_sobrescribir(spark, ruta_actual, fecha_tag):
    """Archiva el Parquet actual en Silver/historico/ antes de sobreescribir."""
    historico = f"{SILVER}/historico/{ruta_actual.split('/')[-1]}_{fecha_tag}"
    try:
        df_prev = spark.read.parquet(ruta_actual)
        df_prev.write.mode("overwrite").parquet(historico)
        print(f"  ✓ Archivado: historico/{ruta_actual.split('/')[-1]}_{fecha_tag}")
    except Exception:
        print(f"  (sin versión anterior para archivar)")

print("✓ Configuración global cargada")
print(f"  BRONZE:  {BRONZE}")
print(f"  TRUSTED: {TRUSTED}")
print(f"  SILVER:  {SILVER}")
print(f"  GROUPS:  {len(GROUPS)} materias")
print(f"  FEATURES: {len(FEATURES_CLASIF)} features")
print(f"  FECHA_TAG: {FECHA_TAG}")

In [0]:
# ============================================================
# NOTEBOOK 04 — EARLY ALERT (v2)
# Vermont School - Early Warning System V2
#
# CAMBIOS vs v1:
# → Lee de predictions_25_26_v2 (nuevo pipeline)
# → 4 categorías (Riesgo Confirmado, Punto Ciego,
#                 Riesgo Teórico, Sin Riesgo)
# → Integra regresión T3 + intervalos P10-P90
# → Genera tabla maestra para Streamlit
#
# INPUT:  silver/predictions_25_26_v2
#         silver/t3_predictions_25_26
#         silver/t3_intervals_25_26
#         silver/clusters_25_26
#         silver/lsc_25_26
#         silver/descriptive/por_estudiante
#
# OUTPUT: silver/early_alerts_v2  ← tabla maestra Streamlit
#         privado/reporte_real_v2 ← con nombres reales
# ============================================================

import pandas as pd
import numpy as np
import os

SILVER  = "/Volumes/workspace/vermont/silver"
PRIVADO = "/Volumes/workspace/vermont/privado"

print("=" * 60)
print("NOTEBOOK 04 — EARLY ALERT v2")
print("Vermont School EWS | 2025-26")
print("=" * 60)

# ── Cargar todos los datasets de Silver ──
print("\n── Cargando datasets ──")

# Predicciones de clasificación (4 categorías)
df_pred = spark.read.parquet(
    f"{SILVER}/predictions_25_26_v2"
).toPandas()
print(f"✓ Clasificación: {len(df_pred)} estudiantes")

# Predicciones T3 regresión
df_t3 = spark.read.parquet(
    f"{SILVER}/t3_predictions_25_26"
).toPandas()
print(f"✓ Regresión T3:  {len(df_t3)} estudiantes")

# Intervalos P10-P90
df_int = spark.read.parquet(
    f"{SILVER}/t3_intervals_25_26"
).toPandas()
print(f"✓ Intervalos:    {len(df_int)} estudiantes")

# Clusters
df_clust = spark.read.parquet(
    f"{SILVER}/clusters_25_26"
).toPandas()
print(f"✓ Clusters:      {len(df_clust)} estudiantes")

# LSC
df_lsc = spark.read.parquet(
    f"{SILVER}/lsc_25_26"
).toPandas()
print(f"✓ LSC:           {len(df_lsc)} estudiantes")

# Ficha por estudiante (descriptivo)
df_desc = spark.read.parquet(
    f"{SILVER}/descriptive/por_estudiante"
).toPandas()
print(f"✓ Descriptivo:   {len(df_desc)} estudiantes")

In [0]:
# CELDA 2 — JOIN Y TABLA MAESTRA

print("=" * 60)
print("CONSTRUYENDO TABLA MAESTRA")
print("=" * 60)

GROUPS = [
    'Science', 'I_and_S', 'Mathematics', 'English',
    'Lengua_Castellana', 'Mandarin', 'Financial_Maths',
    'ICT_STEM', 'Physical_Education', 'Research_Methodology'
]

# ── Base: clasificación ──
df_master = df_pred[[
    'student_id', 'grade', 'section_anon',
    'pred_label', 'proba_critical', 'confianza',
    'n_bajo_acumulada', 't3_confirma_riesgo',
    'categoria', 'modelo', 'fecha_ejecucion'
]].copy()

# ── Join: regresión T3 ──
t3_pred_cols = [c for c in df_t3.columns
                if '_T3_pred' in c and 'confiable' not in c]
t3_conf_cols = [c for c in df_t3.columns if 'confiable' in c]

df_master = df_master.merge(
    df_t3[['student_id'] + t3_pred_cols + t3_conf_cols],
    on='student_id', how='left'
)
print(f"✓ Join regresión T3: {len(t3_pred_cols)} materias")

# ── Join: intervalos P10-P90 ──
p10_cols = [c for c in df_int.columns if '_p10' in c]
p50_cols = [c for c in df_int.columns if '_p50' in c]
p90_cols = [c for c in df_int.columns if '_p90' in c]
amp_cols = [c for c in df_int.columns if '_amp' in c]

df_master = df_master.merge(
    df_int[['student_id'] + p10_cols + p50_cols +
            p90_cols + amp_cols + ['incertidumbre_promedio']],
    on='student_id', how='left'
)
print(f"✓ Join intervalos: P10/P50/P90 por materia")

# ── Join: clusters ──
df_master = df_master.merge(
    df_clust[['student_id', 'cluster', 'perfil',
               'avg_T1', 'avg_T2', 'tendencia_general',
               'n_bajo_T1', 'n_bajo_T2',
               'total_absences', 'n_f1', 'n_f2',
               'indice_disciplinario', 'n_destacadas_T2',
               'min_nota_T2']],
    on='student_id', how='left'
)
print(f"✓ Join clusters: perfil + variables descriptivas")

# ── Join: LSC ──
df_master = df_master.merge(
    df_lsc[['student_id', 'marcador_LSC']],
    on='student_id', how='left'
)
df_master['marcador_LSC'] = (
    df_master['marcador_LSC'].fillna(0).astype(int)
)
print(f"✓ Join LSC: {df_master['marcador_LSC'].sum()} estudiantes")

# ── Join: notas reales T1, T2, T3 parcial ──
# Columnas de notas por trimestre
notas_cols = [f'{g}_{t}' for g in GROUPS
              for t in ['T1', 'T2', 'T3']
              if f'{g}_{t}' in df_desc.columns]

# Notas mínimas T3 necesarias
min_t3_cols = [c for c in df_desc.columns if 'min_T3' in c]

# Variables descriptivas adicionales
desc_extra = [c for c in [
    'pct_asistencia',
    'late', 'early_leave', 'absence_class',
    'total_absences', 'n_f1', 'n_f2'
] if c in df_desc.columns]

cols_desc = ['student_id'] + notas_cols + min_t3_cols + desc_extra
# Evitar columnas duplicadas
cols_desc = list(dict.fromkeys(cols_desc))

df_master = df_master.merge(
    df_desc[cols_desc],
    on='student_id', how='left',
    suffixes=('', '_desc')
)
print(f"✓ Join notas reales: {len(notas_cols)} columnas")

# ── Resumen ──
print(f"\n── Tabla maestra ──")
print(f"  Filas:    {len(df_master)}")
print(f"  Columnas: {len(df_master.columns)}")

print(f"\n── Distribución categorías ──")
for cat in ['Riesgo Confirmado', 'Punto Ciego',
            'Riesgo Teórico', 'Sin Riesgo']:
    n   = (df_master['categoria'] == cat).sum()
    pct = n / len(df_master) * 100
    print(f"  {cat:<22}: {n:>3} ({pct:.1f}%)")

print(f"\n── Distribución perfiles ──")
print(df_master['perfil'].value_counts().to_string())

print(f"\n── LSC por categoría ──")
print(pd.crosstab(df_master['categoria'],
                  df_master['marcador_LSC'],
                  margins=True).to_string())

In [0]:
# CELDA 3 — GUARDAR TABLA MAESTRA Y REPORTE

print("=" * 60)
print("GUARDANDO TABLA MAESTRA")
print("=" * 60)

# ── Guardar en Silver para Streamlit ──
MASTER_PATH = f"{SILVER}/early_alerts_v2"

archivar_antes_de_sobrescribir(spark, MASTER_PATH, FECHA_TAG)
spark.createDataFrame(df_master).write\
    .mode("overwrite")\
    .parquet(MASTER_PATH)

spark.createDataFrame(df_master).write\
    .mode("overwrite")\
    .parquet(MASTER_PATH)

print(f"✓ Tabla maestra guardada: {MASTER_PATH}")
print(f"  {len(df_master)} estudiantes | {len(df_master.columns)} columnas")

# ── Reporte de texto por sección ──
print("\n" + "=" * 60)
print("REPORTE POR SECCIÓN — modo anónimo")
print("=" * 60)

ORDEN_CAT = {
    'Riesgo Confirmado': 0,
    'Punto Ciego':       1,
    'Riesgo Teórico':    2,
    'Sin Riesgo':        3
}

EMOJIS = {
    'Riesgo Confirmado': '🔴',
    'Punto Ciego':       '🟠',
    'Riesgo Teórico':    '🔵',
    'Sin Riesgo':        '🟢'
}

ACCIONES = {
    'Riesgo Confirmado': 'Intervención urgente',
    'Punto Ciego':       'Investigar — modelo no detectó',
    'Riesgo Teórico':    'Monitoreo activo',
    'Sin Riesgo':        'Seguimiento rutinario'
}

for seccion in sorted(df_master['section_anon'].unique()):
    df_sec = df_master[
        df_master['section_anon'] == seccion
    ].copy()
    df_sec['orden'] = df_sec['categoria'].map(ORDEN_CAT)
    df_sec = df_sec.sort_values('orden')

    n_total   = len(df_sec)
    n_atencion = (df_sec['categoria'] != 'Sin Riesgo').sum()

    print(f"\n{'='*60}")
    print(f"SECCIÓN: {seccion} "
          f"({n_total} estudiantes — "
          f"{n_atencion} requieren atención)")
    print(f"{'='*60}")

    df_atencion = df_sec[df_sec['categoria'] != 'Sin Riesgo']
    if len(df_atencion) == 0:
        print("  ✅ Todos en track")
        continue

    for _, row in df_atencion.iterrows():
        cat   = row['categoria']
        emoji = EMOJIS[cat]
        accion = ACCIONES[cat]
        lsc_tag = ' [LSC]' if row['marcador_LSC'] == 1 else ''
        perfil_tag = f" | {row['perfil']}" if pd.notna(
            row.get('perfil')) else ''

        print(f"\n  {emoji} {cat}{lsc_tag}{perfil_tag}")
        print(f"  Estudiante:  {row['student_id']}")
        print(f"  Acción:      {accion}")
        print(f"  Confianza modelo: {row['confianza']*100:.0f}%")
        print(f"  Materias bajo 4.0 (acumulada): "
              f"{int(row['n_bajo_acumulada'])}")

        # Materias en riesgo con predicción T3
        print(f"  Predicciones T3 por materia:")
        for g in GROUPS:
            t3_pred = f'{g}_T3_pred'
            t3_p10  = f'{g}_T3_p10'
            t3_p90  = f'{g}_T3_p90'
            min_t3  = f'{g}_min_T3'
            t3_real = f'{g}_T3'
            bajo    = f'{g}_bajo'

            if t3_pred not in row.index:
                continue

            pred_val = row.get(t3_pred, np.nan)
            p10_val  = row.get(t3_p10, np.nan)
            p90_val  = row.get(t3_p90, np.nan)
            min_val  = row.get(min_t3, np.nan)
            real_val = row.get(t3_real, np.nan)
            es_bajo  = row.get(bajo, False)

            if pd.isna(pred_val):
                continue

            # Mostrar solo materias en riesgo o con pred baja
            if es_bajo or pred_val < 4.0:
                icon = '⚠️' if pred_val < 4.0 else '📊'
                real_str = (f" | T3 parcial: {real_val:.2f}"
                            if not pd.isna(real_val) else "")
                min_str  = (f" | Necesita ≥{min_val:.2f}"
                            if not pd.isna(min_val) else "")
                print(f"    {icon} {g:<22} "
                      f"Pred: {pred_val:.2f} "
                      f"[{p10_val:.1f}-{p90_val:.1f}]"
                      f"{real_str}{min_str}")
        print()

# ── Guardar reporte como CSV en Silver ──
reporte_cols = [
    'student_id', 'grade', 'section_anon',
    'categoria', 'pred_label', 'confianza',
    'n_bajo_acumulada', 'marcador_LSC',
    'perfil', 'cluster',
    'avg_T1', 'avg_T2', 'tendencia_general',
    'total_absences', 'n_f1', 'n_f2',
    'incertidumbre_promedio'
] + t3_pred_cols + p10_cols + p90_cols + min_t3_cols

# Solo columnas que existen
reporte_cols = [c for c in reporte_cols
                if c in df_master.columns]
reporte_cols = list(dict.fromkeys(reporte_cols))

df_reporte = df_master[reporte_cols].copy()

archivar_antes_de_sobrescribir(spark, f"{SILVER}/reporte_alertas_v2", FECHA_TAG)
spark.createDataFrame(df_reporte).write\
    .mode("overwrite")\
    .parquet(f"{SILVER}/reporte_alertas_v2")

print(f"\n✓ Reporte guardado: {SILVER}/reporte_alertas_v2")
print(f"  {len(df_reporte)} estudiantes | "
      f"{len(df_reporte.columns)} columnas")

print(f"\n{'='*60}")
print(f"NOTEBOOK 04 COMPLETO ✓")
print(f"{'='*60}")
print(f"  Tabla maestra:  {MASTER_PATH}")
print(f"  Reporte:        {SILVER}/reporte_alertas_v2")
print(f"\n  Listo para Streamlit ✓")
print(f"  Listo para 00_deanonymizer (actualizar) ✓")

In [0]:
# CELDA FINAL — Exportar CSV + Snapshots + Push a GitHub
# v3: fecha_tag desde datos (no desde ejecución), snapshot siempre sobreescribe
import requests, base64, json, os, re
from datetime import datetime
import pandas as pd

SILVER  = "/Volumes/workspace/vermont/silver"
BRONZE  = "/Volumes/workspace/vermont/bronze"
RAW_25  = f"{BRONZE}/raw/25_26"

# ── Cargar datos ──
df_export = spark.read.parquet(f"{SILVER}/early_alerts_v2").toPandas()

df_pred_extra = spark.read.parquet(
    f"{SILVER}/predictions_25_26_v2"
).select("student_id", "proba_critical", "proba_recovery", "proba_riesgo")\
 .toPandas()

df_export = df_export.merge(
    df_pred_extra[["student_id", "proba_critical", "proba_recovery", "proba_riesgo"]],
    on="student_id", how="left"
)
if 'proba_critical_x' in df_export.columns:
    df_export = df_export.rename(columns={'proba_critical_x': 'proba_critical'})
    df_export = df_export.drop(columns=['proba_critical_y'])

print("── Verificación ──")
print([c for c in df_export.columns if 'proba' in c])
print(f"proba_riesgo: {df_export['proba_riesgo'].min():.3f} – {df_export['proba_riesgo'].max():.3f}")
print("── Distribución de categorías ──")
print(df_export["categoria"].value_counts())

# ── Detectar fecha de corte desde los datos (no desde ejecución) ──
def fecha_corte_datos(carpeta):
    """
    Lee la fecha del archivo XLS más reciente en raw/25_26.
    Lanza error si no hay archivos con fecha en el nombre —
    así evitamos snapshots con fecha incorrecta.
    """
    archivos = [f for f in os.listdir(carpeta) if f.endswith('.xls')]
    fechas = []
    for f in archivos:
        m = re.search(r'_(\d{2})_(\d{2})_(\d{2})\.xls$', f)
        if m:
            d, mo, y = m.groups()
            fechas.append((datetime(int(f"20{y}"), int(mo), int(d)), f"{d}_{mo}_{y}"))
    if not fechas:
        raise ValueError(
            "⚠️ No se encontró ningún archivo con fecha en el nombre en raw/25_26. "
            "Asegúrate de que los archivos tengan formato _DD_MM_AA.xls antes de correr el pipeline."
        )
    return max(fechas, key=lambda x: x[0])

fecha_dt, fecha_tag = fecha_corte_datos(RAW_25)
fecha_display = fecha_dt.strftime("%d de %B de %Y")
print(f"\n── Fecha de corte de los datos: {fecha_display} ({fecha_tag}) ──")

# Agregar fecha al CSV
df_export["fecha_corte"] = fecha_display
df_export["fecha_corte_tag"] = fecha_tag

# ── Guardar CSV principal (siempre sobreescribe) ──
OUTPUT_CSV = f"{BRONZE}/dashboard_data.csv"
df_export.to_csv(OUTPUT_CSV, index=False)
print(f"✓ CSV principal: {len(df_export)} filas × {len(df_export.columns)} cols")

# ── Guardar snapshot (siempre sobreescribe — refleja datos + pipeline más recientes) ──
SNAPSHOTS_DIR = f"{BRONZE}/snapshots"
os.makedirs(SNAPSHOTS_DIR, exist_ok=True)
SNAPSHOT_CSV = f"{SNAPSHOTS_DIR}/dashboard_data_{fecha_tag}.csv"
df_export.to_csv(SNAPSHOT_CSV, index=False)
print(f"✓ Snapshot: dashboard_data_{fecha_tag}.csv (sobreescrito si existía)")

# ── Actualizar índice de snapshots ──
snapshots = sorted([
    f for f in os.listdir(SNAPSHOTS_DIR)
    if f.startswith("dashboard_data_") and f.endswith(".csv")
])
index_rows = []
for s in snapshots:
    tag = s.replace("dashboard_data_", "").replace(".csv", "")
    try:
        dt = datetime.strptime(tag, "%d_%m_%y")
        display = dt.strftime("%d de %B de %Y")
    except Exception:
        display = tag
    index_rows.append({"archivo": s, "fecha_tag": tag, "fecha_display": display})

df_index = pd.DataFrame(index_rows)
INDEX_CSV = f"{SNAPSHOTS_DIR}/index.csv"
df_index.to_csv(INDEX_CSV, index=False)
print(f"✓ Índice actualizado: {len(df_index)} snapshot(s)")
print(df_index[["archivo", "fecha_display"]].to_string(index=False))

# ── Push a GitHub ──
with open(f"{PRIVADO}/github_token.txt") as f:
    GITHUB_TOKEN = f.read().strip()

headers_gh = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}

archivos_push = [
    (OUTPUT_CSV,   "notebooks_v2/dashboard_data.csv"),
    (SNAPSHOT_CSV, f"notebooks_v2/snapshots/dashboard_data_{fecha_tag}.csv"),
    (INDEX_CSV,    "notebooks_v2/snapshots/index.csv"),
]

for local_path, github_path in archivos_push:
    with open(local_path, "rb") as f:
        content_b64 = base64.b64encode(f.read()).decode("utf-8")

    api_url = f"https://api.github.com/repos/afvh3146/vermont-alerta-temprana/contents/{github_path}"
    r = requests.get(api_url, headers=headers_gh)
    sha = r.json().get("sha", None)

    payload = {
        "message": f"chore: datos {fecha_tag} — pipeline actualizado",
        "content": content_b64,
        "branch": "main"
    }
    if sha:
        payload["sha"] = sha

    r2 = requests.put(api_url, headers=headers_gh, data=json.dumps(payload))
    if r2.status_code in [200, 201]:
        print(f"✓ Push: {github_path}")
    else:
        print(f"✗ Error {r2.status_code}: {github_path} — {r2.json().get('message','')}")

In [0]:
import pandas as pd

df = pd.read_csv("/Volumes/workspace/vermont/bronze/dashboard_data.csv")
print(df["n_bajo_acumulada"].value_counts().sort_index())

In [0]:
import joblib
import json
import pandas as pd

SILVER = "/Volumes/workspace/vermont/silver"

# Cargar modelos
clf = joblib.load(f"{SILVER}/models/rf_classifier.joblib")
reg = joblib.load(f"{SILVER}/models/rf_regressor.joblib")

# Cargar listas de features
with open(f"{SILVER}/models/feature_list.json") as f:
    features_clf = json.load(f)

with open(f"{SILVER}/models/feature_list_reg.json") as f:
    features_reg = json.load(f)

with open(f"{SILVER}/models/target_list_reg.json") as f:
    targets_reg = json.load(f)

# Feature importance clasificador
df_imp_clf = pd.DataFrame({
    'Feature':    features_clf,
    'Importance': clf.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

print("── Feature importance CLASIFICADOR (top 15) ──")
print(df_imp_clf.to_string(index=False))

# Feature importance regresor
df_imp_reg = pd.DataFrame({
    'Feature':    features_reg,
    'Importance': reg.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

print("\n── Feature importance REGRESOR (top 15) ──")
print(df_imp_reg.to_string(index=False))

In [0]:
### Celda de verificación, para correr después del job

import os
import pandas as pd

SILVER = "/Volumes/workspace/vermont/silver"
BRONZE = "/Volumes/workspace/vermont/bronze"

# ── Histórico en Silver ──
print("── Histórico en Silver ──")
historico_path = f"{SILVER}/historico"
if os.path.exists(historico_path):
    for f in sorted(os.listdir(historico_path)):
        print(f"  {f}")
else:
    print("  (vacío — primera corrida)")

# ── Snapshots en Bronze ──
print("\n── Snapshots en Bronze ──")
for f in sorted(os.listdir(f"{BRONZE}/snapshots")):
    print(f"  {f}")

# ── Distribución actual ──
print("\n── Distribución actual ──")
df = pd.read_csv(f"{BRONZE}/dashboard_data.csv")
print(f"  Fecha corte: {df['fecha_corte'].iloc[0]}")
print(f"  Estudiantes: {len(df)}")
print(df["categoria"].value_counts().to_string())

# ── Distribución por grado ──
print("\n── Por grado y categoría ──")
print(df.groupby(["grade", "categoria"]).size().unstack(fill_value=0).to_string())

# ── Verificar T3 por grado ──
print("\n── T3 con datos por grado ──")
t3_cols = [c for c in df.columns if '_T3' in c and 'pred' not in c 
           and 'p10' not in c and 'p90' not in c and 'min' not in c]
for grado in sorted(df["grade"].unique()):
    sub = df[df["grade"] == grado]
    con_t3 = sub[t3_cols].notna().any(axis=1).sum()
    print(f"  Grado {grado}: {con_t3}/{len(sub)} estudiantes con T3")